# _____NHS Scotland Prescription Analysis – August 2025______


In [278]:
# Step 1:  Install or Upgrade the Folium Library for geospatial mapping.

!pip install folium --upgrade  


In [279]:
# Step 2:  Installing Folium Using the Correct Jupyter Python Environment

import sys
!{sys.executable} -m pip install folium


In [280]:
# Step 3: Install Plotly
import sys
!{sys.executable} -m pip install plotly

In [281]:
# Step 3: Import required Libraries
import pandas as pd   # For data handling
import folium         # For interactive maps
import plotly.express as px   # For visualizations
import numpy as np



In [282]:
# Step 4: Read the dataset into a pandas DataFrame.
file_path="/Users/kishorravi/Documents/VIDYA/DATA SCIENCE/Python/Data Visualization/PROJECTS/prescription.csv"
df_prescriber_loc=pd.read_csv(file_path)
print("Data readed in to panda's data frame!")

Data readed in to panda's data frame!


# ****** Exploratory Data Analysis ******

In [283]:

# Step 5: Preview the First Five Rows of the Dataset
df_prescriber_loc.head()

,HBT,GPPractice,DMDCode,BNFItemCode,BNFItemDescription,PrescribedType,NumberOfPaidItems,PaidQuantity,GrossIngredientCost,PaidDateMonth
0,S08000015,80005,NaN,NaN,NaN,AMP,1,224.00,62.06,202508
1,S08000015,80005,NaN,NaN,NaN,VMP,13,"1,837.00","1,020.02",202508
2,S08000015,80005,NaN,NaN,NaN,VMPP,4,0.00,0.00,202508
3,S08000015,80005,"1,001,411,000,001,102.00",1001010P0AAAHAH,NAPROXEN 250MG GASTRO-RESISTANT TABLETS,VMP,2,77.00,11.07,202508
4,S08000015,80005,"1,001,811,000,001,100.00",1310012F0AAABAB,FUSIDIC ACID 2% CREAM,VMP,2,30.00,3.84,202508


In [284]:
# Step 3: See all column names
df_prescriber_loc.columns

Index(['HBT', 'GPPractice', 'DMDCode', 'BNFItemCode', 'BNFItemDescription',
       'PrescribedType', 'NumberOfPaidItems', 'PaidQuantity',
       'GrossIngredientCost', 'PaidDateMonth'],
      dtype='object')

*** Note:


⭐ Understanding Your Dataset

1️⃣ HBT — Health Board Territory
This tells which NHS Health Board the prescription belongs to.

2️⃣ GPPractice — GP Practice Code
Every GP practice has a unique code.
Example:
80005

3️⃣ DMDCode — Medicine Code
This is a code for the medicine in the NHS Dictionary of Medicines & Devices (DM+D).
Example:
1001411e+15

4️⃣ BNFItemCode — Drug Code (BNF = British National Formulary)
Another code used to identify the specific medicine.
Example:
1001010P0AAAHAH → Naproxen
1310012P0AAAABAB → Fusidic Acid Cream
👉 The BNFItemDescription column tells the actual name.

5️⃣ BNFItemDescription — Medicine Name
This is the real human-readable name of the medicine.
Examples:
“NAPROXEN 250MG GASTRO-RESISTANT TABLETS”

6️⃣ PrescribedType — VMP or AMP
VMP = Virtual Medicinal Product
(generic or grouped medicine)

AMP = Actual Medicinal Product
(specific brand)
Example:
Paracetamol 500mg (generic) → VMP
“Panadol 500mg tablets” (brand) → AMP

7️⃣ NumberOfPaidItems
This is the number of times the medicine was prescribed.
Example:
1 → prescribed once
77 → 77 prescriptions
1837 → 1837 prescriptions of same item

8️⃣ PaidQuantity
How many units were dispensed.
Example:
30 tablets
100ml

9️⃣ GrossIngredientCost
💰 The cost to NHS of this medicine item.
Example:
62.06 = £62.06

🔟 PaidDateMonth — Year & Month
The month of the dataset.
Example:
202508 = August 2025


In [285]:
# Step 4: Check the Structre
df_prescriber_loc.shape

(1240929, 10)

In [286]:
# Step 5: Check basic info (data types, missing values, memory use)
df_prescriber_loc.info

<bound method DataFrame.info of                HBT  GPPractice                  DMDCode      BNFItemCode  \
0        S08000015       80005                      NaN              NaN   
1        S08000015       80005                      NaN              NaN   
2        S08000015       80005                      NaN              NaN   
3        S08000015       80005 1,001,411,000,001,102.00  1001010P0AAAHAH   
4        S08000015       80005 1,001,811,000,001,100.00  1310012F0AAABAB   
...            ...         ...                      ...              ...   
1240924     SB0806       99995   969,811,000,001,108.00  0209000A0AAABAB   
1240925     SB0806       99995   970,711,000,001,102.00  0106040G0AAAAAA   
1240926     SB0806       99995 9,709,511,000,001,108.00  0407010H0AAAAAA   
1240927     SB0806       99995   976,011,000,001,101.00  0702020F0AAAFAF   
1240928     SB0806       99995   987,611,000,001,101.00  0407010F0AAAAAA   

                              BNFItemDescription Prescr

***  Note: 
  
   this output, we understand:_____

✔ The dataset is very large

✔ It contains rich information about prescriptions

✔ Many columns have missing values

✔ Column types need cleaning


In [287]:
# Step 6: Check for missing values
df_prescriber_loc.isna().sum()

HBT                       0
GPPractice                0
DMDCode                1515
BNFItemCode            3502
BNFItemDescription     3502
PrescribedType            0
NumberOfPaidItems         0
PaidQuantity              0
GrossIngredientCost       0
PaidDateMonth             0
dtype: int64

***  Note: BNFItemDescription	3502	Medicine NAME missing — important column

In [288]:
# Step 7: Remove rows where medicine name is missing
df_prescriber_loc_clean=df_prescriber_loc.dropna(subset=['BNFItemDescription'])
df_prescriber_loc_clean.shape

(1237427, 10)

In [289]:
# Step 8:  — Rename important columns , it makes dashboard clean
df_prescriber_loc_clean=df_prescriber_loc_clean.rename(columns={
    'BNFItemDescription': 'Medicine',
    'NumberOfPaidItems': 'Items',
    'GrossIngredientCost': 'Cost',
    'PaidQuantity': 'Quantity',
    'GPPractice': 'Practice',
    'HBT': 'HealthBoard'})
df_prescriber_loc_clean.columns

Index(['HealthBoard', 'Practice', 'DMDCode', 'BNFItemCode', 'Medicine',
       'PrescribedType', 'Items', 'Quantity', 'Cost', 'PaidDateMonth'],
      dtype='object')

In [290]:
# Step 9: Check descriptive statistics
df_prescriber_loc_clean.describe()


,Practice,DMDCode,Items,Quantity,Cost,PaidDateMonth
count,"1,237,427.00","1,237,427.00","1,237,427.00","1,237,427.00","1,237,427.00","1,237,427.00"
mean,"52,070.85","13,191,874,521,162,354.00",7.72,767.57,87.86,"202,508.00"
std,"25,392.00","14,181,100,896,459,730.00",28.08,"6,147.06","1,017.59",0.00
min,"10,002.00","301,011,000,001,105.00",1.00,0.00,0.00,"202,508.00"
25%,"30,261.00","1,287,911,000,001,106.00",1.00,40.00,9.30,"202,508.00"
50%,"52,344.00","4,499,211,000,001,101.00",2.00,112.00,25.02,"202,508.00"
75%,"76,052.00","22,820,911,000,001,108.00",5.00,392.00,72.00,"202,508.00"
max,"99,999.00","45,419,211,000,001,112.00","4,354.00","1,867,551.00","891,502.75","202,508.00"


In [291]:
# Step 10: Make describe() output easy to read
pd.set_option('display.float_format', '{:,.2f}'.format)
df_prescriber_loc_clean.describe()


,Practice,DMDCode,Items,Quantity,Cost,PaidDateMonth
count,"1,237,427.00","1,237,427.00","1,237,427.00","1,237,427.00","1,237,427.00","1,237,427.00"
mean,"52,070.85","13,191,874,521,162,354.00",7.72,767.57,87.86,"202,508.00"
std,"25,392.00","14,181,100,896,459,730.00",28.08,"6,147.06","1,017.59",0.00
min,"10,002.00","301,011,000,001,105.00",1.00,0.00,0.00,"202,508.00"
25%,"30,261.00","1,287,911,000,001,106.00",1.00,40.00,9.30,"202,508.00"
50%,"52,344.00","4,499,211,000,001,101.00",2.00,112.00,25.02,"202,508.00"
75%,"76,052.00","22,820,911,000,001,108.00",5.00,392.00,72.00,"202,508.00"
max,"99,999.00","45,419,211,000,001,112.00","4,354.00","1,867,551.00","891,502.75","202,508.00"


*** Note: 

Descriptive Statistics – Short Summary

The summary above shows the typical values in the dataset:

Items: Most prescriptions have 1–5 items, with a median of 2. A few rows show very large values (max 4,354), indicating outliers.

Quantity: Normal range is 40–392 units, median 112. Extremely large maximum values suggest special or bulk supplies.

Cost: Most prescriptions cost £9–£72, median £25. Some medicines are very expensive (up to £891k).

PaidDateMonth: All entries belong to August 2025.

  # *** Basic Stats (KPI EDA)   ****

In [292]:
# Step 11: Total prescriptions:
df_prescriber_loc_clean['Items'].sum()

9553410

*** Note: NHS Scotland dispensed 9.55 million items in August 2025.

In [293]:
# Step 12: Total NHS Spend (Total NHS Expenditure on Prescriptions)
df_prescriber_loc_clean['Cost'].sum()

108715200.35000004

*** Note: NHS spent £108.7 million in August 2025 on prescriptions.

In [294]:
# Step 13: Unique Medicines (Number of Distinct Medicines Prescribed)
df_prescriber_loc_clean['Medicine'].nunique()

13247

*** Note: 13.2k different medicines were prescribed across Scotland.

In [295]:
# Step 14: Unique GP Practices (Number of Distinct GP Practices)
df_prescriber_loc_clean['Practice'].nunique()

1070

*** Note: Data contains prescriptions from 1,070 GP practices.

# ******  Building Analytical Table (Deep EDA) ***

In [296]:
# Step 15: Top 10 medicines by number of items

top10_items = (
    df_prescriber_loc_clean.groupby('Medicine')['Items']
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

top10_items


Medicine
OMEPRAZOLE 20MG GASTRO-RESISTANT CAPSULES         315409
CO-CODAMOL 30MG/500MG CAPLETS                     157583
PARACETAMOL 500MG CAPLETS                         157250
ATORVASTATIN 20MG TABLETS                         138453
AMLODIPINE 5MG TABLETS                            135612
ASPIRIN 75MG DISPERSIBLE TABLETS                  126079
SALBUTAMOL 100MICROGRAMS/DOSE INHALER CFC FREE    116238
LANSOPRAZOLE 30MG GASTRO-RESISTANT CAPSULES       103350
ATORVASTATIN 40MG TABLETS                          91248
AMLODIPINE 10MG TABLETS                            86421
Name: Items, dtype: int64

*** Note:
The table above shows the medicines with the highest number of prescription items in August 2025.
A prescription item represents one prescription issued, regardless of the quantity dispensed.
These top medicines indicate the most commonly used treatments across Scotland.

In [297]:
# Step 16:  Top 10 medicines by cost
top10_cost = (
    df_prescriber_loc_clean.groupby('Medicine')['Cost']
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

top10_cost

Medicine
FREESTYLE LIBRE 2 PLUS SENSOR                                 2,925,975.00
FORXIGA 10MG TABLETS                                          1,889,325.88
GLECAPREVIR 100MG / PIBRENTASVIR 40MG TABLETS                 1,296,117.59
XTANDI 40MG TABLETS                                           1,265,175.96
TRELEGY ELLIPTA 92MICROG/55MICROG/22MICROG/DOSE DRY PDR INH   1,254,455.00
TRIMBOW 87MICROG/DOSE / 5MICROG/DOSE / 9MICROG/DOSE INH       1,056,296.50
LIXIANA 60MG TABLETS                                            843,174.52
ANORO ELLIPTA 55MICROG/DOSE / 22MICROG/DOSE DRY PDR INH         774,670.00
OMEPRAZOLE 20MG GASTRO-RESISTANT CAPSULES                       747,660.55
DAPAGLIFLOZIN 10MG TABLETS                                      736,435.60
Name: Cost, dtype: float64

*** Note:
'The list above shows the medicines that generated the highest overall spending for NHS Scotland in August 2025.

# *****.  Health Board Analysis *****

In [298]:
# Step 17: Total Number of Prescribed Items per NHS Health Board
items_by_board = (
    df_prescriber_loc_clean.groupby('HealthBoard')['Items']
    .sum()
    .sort_values(ascending=False)
)

items_by_board


HealthBoard
S08000031    2204837
S08000032    1312647
S08000024    1226834
S08000020     875672
S08000015     773643
S08000030     708479
S08000029     645322
S08000022     580048
S08000019     543310
S08000017     337104
S08000016     210707
S08000028      59146
S08000026      42032
S08000025      33421
SB0806           208
Name: Items, dtype: int64

*** Note: The output shows the total number of prescription items issued by each NHS Health Board in Scotland.
A prescription item represents one prescription written, regardless of quantity.

In [299]:
# Step 18: Total NHS Cost per Health Board
cost_by_board = (
    df_prescriber_loc_clean.groupby('HealthBoard')['Cost']
    .sum()
    .sort_values(ascending=False)
)

cost_by_board


HealthBoard
S08000031   25,432,790.58
S08000024   15,352,296.83
S08000032   13,896,538.80
S08000020   10,304,560.72
S08000015    8,339,934.78
S08000030    8,272,551.05
S08000029    6,989,483.67
S08000022    6,966,157.20
S08000019    6,223,564.05
S08000017    3,193,033.40
S08000016    2,334,712.67
S08000028      563,851.73
S08000026      450,357.98
S08000025      394,121.32
SB0806           1,245.57
Name: Cost, dtype: float64

*** Note: This table shows the total cost of all prescriptions issued in each NHS Health Board.
It highlights where NHS Scotland spends the most on medicines.

In [300]:
# Step 19: Find high-cost medicines with low usage
high_cost_low_qty = df_prescriber_loc_clean[
    (df_prescriber_loc_clean['Quantity'] < 5) &
    (df_prescriber_loc_clean['Cost'] > 200)
]

high_cost_low_qty[['Medicine','Quantity','Cost']]\
    .sort_values('Cost', ascending=False)\
    .head(20)


,Medicine,Quantity,Cost
384036,SIGNIFOR 40MG INJ VIALS,2.00,"4,600.00"
611485,LANREOTIDE 120MG/0.5ML INJ PRE-FILLED SYRINGES,4.00,"3,748.00"
257965,LANREOTIDE 120MG/0.5ML INJ PRE-FILLED SYRINGES,4.00,"2,998.40"
247816,LANREOTIDE 120MG/0.5ML INJ PRE-FILLED SYRINGES,4.00,"2,998.40"
725877,LANREOTIDE 120MG/0.5ML INJ PRE-FILLED SYRINGES,4.00,"2,998.40"
312286,LANREOTIDE 120MG/0.5ML INJ PRE-FILLED SYRINGES,3.00,"2,811.00"
514986,SOMATULINE AUTOGEL 120MG/0.5ML INJ PFS WITH SA...,3.00,"2,811.00"
162422,LANREOTIDE 120MG/0.5ML INJ PRE-FILLED SYRINGES,3.00,"2,811.00"
674823,LANREOTIDE 120MG/0.5ML INJ PRE-FILLED SYRINGES,3.00,"2,811.00"
113078,LANREOTIDE 120MG/0.5ML INJ PRE-FILLED SYRINGES,3.00,"2,811.00"


*** Note: The output highlights medicines that were dispensed in very small quantities (less than 5 units) but still had a very high total cost (over £2,000 per prescription).

# ***** Visualizations with Maps *****

In [301]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# -----------------------------
# Calculate KPIs
# -----------------------------
total_items = int(df_prescriber_loc_clean["Items"].sum())
total_cost = float(df_prescriber_loc_clean["Cost"].sum())
unique_meds = df_prescriber_loc_clean["Medicine"].nunique()
unique_practices = df_prescriber_loc_clean["Practice"].nunique()

# -----------------------------
# Create KPI layout (2 x 2)
# -----------------------------
fig_kpi = make_subplots(
    rows=2, cols=2,
    specs=[[{"type": "indicator"}, {"type": "indicator"}],
           [{"type": "indicator"}, {"type": "indicator"}]]
)

# Card colours
card_colours = ["#E3F2FD", "#FFF3E0", "#E8F5E9", "#F3E5F5"]  # light blue, orange, green, purple
number_colours = ["#0D47A1", "#E65100", "#1B5E20", "#4A148C"]

# KPI 1 – Total Items
fig_kpi.add_trace(
    go.Indicator(
        mode="number",
        value=total_items,
        title={"text": "💊 Total Prescriptions (Items)", "font": {"size": 16}},
        number={"valueformat": ",", "font": {"size": 42, "color": number_colours[0]}},
    ),
    row=1, col=1
)

# KPI 2 – Total Cost
fig_kpi.add_trace(
    go.Indicator(
        mode="number",
        value=total_cost,
        title={"text": "💷 Total NHS Spend on Prescriptions", "font": {"size": 16}},
        number={"prefix": "£", "valueformat": ",.0f",
                "font": {"size": 42, "color": number_colours[1]}},
    ),
    row=1, col=2
)

# KPI 3 – Unique Medicines
fig_kpi.add_trace(
    go.Indicator(
        mode="number",
        value=unique_meds,
        title={"text": "🧪 Unique Medicines Prescribed", "font": {"size": 16}},
        number={"valueformat": ",", "font": {"size": 42, "color": number_colours[2]}},
    ),
    row=2, col=1
)

# KPI 4 – Unique GP Practices
fig_kpi.add_trace(
    go.Indicator(
        mode="number",
        value=unique_practices,
        title={"text": "🏥 Unique GP Practices", "font": {"size": 16}},
        number={"valueformat": ",", "font": {"size": 42, "color": number_colours[3]}},
    ),
    row=2, col=2
)

# -----------------------------
# Add coloured "cards" behind each KPI
# (approximate domains for a 2x2 grid)
# -----------------------------
fig_kpi.add_shape(type="rect", x0=0.00, x1=0.48, y0=0.52, y1=1.00,
                  fillcolor=card_colours[0], layer="below", line_width=0)
fig_kpi.add_shape(type="rect", x0=0.52, x1=1.00, y0=0.52, y1=1.00,
                  fillcolor=card_colours[1], layer="below", line_width=0)
fig_kpi.add_shape(type="rect", x0=0.00, x1=0.48, y0=0.00, y1=0.48,
                  fillcolor=card_colours[2], layer="below", line_width=0)
fig_kpi.add_shape(type="rect", x0=0.52, x1=1.00, y0=0.00, y1=0.48,
                  fillcolor=card_colours[3], layer="below", line_width=0)

# -----------------------------
# Layout styling
# -----------------------------
fig_kpi.update_layout(
    title="<b>Basic Prescription KPIs – NHS Scotland (EDA Overview)</b>",
    title_x=0.5,
    title_font=dict(size=22),
    height=600,
    margin=dict(l=40, r=40, t=80, b=40),
    paper_bgcolor="#F5F5F5"  # light grey dashboard background
)

fig_kpi.show()


In [302]:

#  Step 20: Combine your existing summaries into a single DataFrame ---
board_summary = pd.DataFrame({
    "HealthBoard": items_by_board.index,
    "TotalItems": items_by_board.values,
    "TotalCost": cost_by_board.reindex(items_by_board.index).values
})

board_summary["CostPerItem"] = (
    board_summary["TotalCost"] / board_summary["TotalItems"]
)

# --- 2️⃣ Coordinates for each NHS Scotland Health Board ---
board_coords = pd.DataFrame([
    {"HealthBoard": "S08000015", "HBName": "NHS Ayrshire & Arran",      "lat": 55.45, "lon": -4.63},
    {"HealthBoard": "S08000016", "HBName": "NHS Borders",               "lat": 55.60, "lon": -2.72},
    {"HealthBoard": "S08000017", "HBName": "NHS Dumfries & Galloway",   "lat": 55.07, "lon": -3.61},
    {"HealthBoard": "S08000018", "HBName": "NHS Fife",                  "lat": 56.12, "lon": -3.16},
    {"HealthBoard": "S08000019", "HBName": "NHS Forth Valley",          "lat": 56.00, "lon": -3.80},
    {"HealthBoard": "S08000020", "HBName": "NHS Grampian",              "lat": 57.15, "lon": -2.11},
    {"HealthBoard": "S08000021", "HBName": "NHS Greater Glasgow & Clyde","lat": 55.86, "lon": -4.25},
    {"HealthBoard": "S08000022", "HBName": "NHS Highland",              "lat": 57.48, "lon": -4.22},
    {"HealthBoard": "S08000023", "HBName": "NHS Lanarkshire",           "lat": 55.77, "lon": -4.05},
    {"HealthBoard": "S08000024", "HBName": "NHS Lothian",               "lat": 55.95, "lon": -3.19},
    {"HealthBoard": "S08000025", "HBName": "NHS Orkney",                "lat": 58.98, "lon": -2.96},
    {"HealthBoard": "S08000026", "HBName": "NHS Shetland",              "lat": 60.15, "lon": -1.15},
    {"HealthBoard": "S08000027", "HBName": "NHS Tayside",               "lat": 56.46, "lon": -2.97},
    {"HealthBoard": "S08000028", "HBName": "NHS Western Isles",         "lat": 58.21, "lon": -6.38},
])

# Merge with your stats
board_map_df = board_summary.merge(board_coords, on="HealthBoard", how="left")


In [303]:
# Step 21: Top 10 Medicines — Usage vs Cost
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        "💊 Top 10 Medicines by Items Dispensed",
        "💷 Top 10 Medicines by Total Cost (£)"
    ]
)

# Chart 1 — Items
fig.add_trace(
    go.Bar(
        x=df_items["Medicine"],
        y=df_items["Items"],
        marker=dict(color="#2E8B57"),
        hovertemplate="Items: %{y:,}<br>Medicine: %{x}"
    ),
    row=1, col=1
)

# Chart 2 — Cost
fig.add_trace(
    go.Bar(
        x=df_cost["Medicine"],
        y=df_cost["Cost"],
        marker=dict(color="#C71585"),
        hovertemplate="Cost: £%{y:,.0f}<br>Medicine: %{x}"
    ),
    row=1, col=2
)

fig.update_layout(
    title_text="<b>Top 10 Medicines — Usage vs Cost</b>",
    title_x=0.5,
    height=750,            # increased to reduce crowding
    showlegend=False,
    margin=dict(l=250, r=40, t=80, b=60)  # increased left margin
)

# Format y-axis
fig.update_yaxes(tickformat=",", row=1, col=1)
fig.update_yaxes(tickformat="£,.0f", row=1, col=2)

# Rotate x-axis labels
fig.update_xaxes(
    tickangle=45,
    automargin=True
)

fig.show()


In [304]:
#Step 21: Interactive  Maps (plotly)— Prescription Cost  by NHS Health Board

fig_cost_bar = px.bar(
    cost_board_df,
    x="Cost",
    y="HealthBoard",
    orientation="h",
    title="<b>Total NHS Prescription Cost by Health Board (£)</b>",
    color="Cost",
    hover_name="HBName",
    color_continuous_scale="Plasma"
)

fig_cost_bar.update_layout(
    xaxis_title="Total Cost (£)",
    yaxis_title="Health Board",
    title_font=dict(size=22),
    height=600,
    margin=dict(l=140)     # extra space on the left
)

# Move the y-axis title further left
fig_cost_bar.update_yaxes(
    title_standoff=40      # try 40–60 if you want more space
)

# Format x-axis as £ with commas
fig_cost_bar.update_xaxes(
    tickformat="£,.0f"
)

fig_cost_bar.show()


In [305]:
#21.2: Map based on Total Items
# --- Beautiful bubble map for Total Items ---

# customdata holds the true values for hover text
items_customdata = board_map_df[["TotalItems", "TotalCost"]].to_numpy()

fig_items = px.scatter_geo(
    board_map_df,
    lat="lat",
    lon="lon",
    hover_name="HBName",
    size="TotalItems",
    color="TotalItems",
    size_max=200,                     # Larger bubbles for visibility
    color_continuous_scale="Viridis",
    projection="mercator",
    title="💊 NHS Scotland – Prescription Items Dispensed by Health Board",
)

# Tooltip styling — clean and professional
fig_items.update_traces(
    customdata=items_customdata,
    hovertemplate=(
        "<b>%{hovertext}</b><br>"
        "Total Items: %{customdata[0]:,.0f}<br>"
        "Total Cost: £%{customdata[1]:,.0f}"
        "<extra></extra>"
    )
)

# Better map framing + land visible
fig_items.update_geos(
    fitbounds="locations",
    visible=False,
    projection_scale=10,
    center={"lat": 56.8, "lon": -4.0},
    showland=True,
    landcolor="#f5f5f5",
    coastlinecolor="#999999",
)

fig_items.update_layout(
    title={
        "text": "<b>💊 NHS Scotland – Prescription Items Dispensed by Health Board</b>",
        "y": 0.95,
        "x": 0.5,
        "xanchor": "center",
        "yanchor": "top",
        "font": dict(size=26)
    },
    width=1000,
    height=900,
    margin=dict(l=0, r=0, t=90, b=0)
)



fig_items.show()


In [306]:
# Step 22: NHS Scotland Health Board Analysis: Cost, Usage, and Outlier Medicines.

# 22.1: Aggregate per Board + Medicine
high_cost_low_summary = (
    high_cost_low_qty
    .groupby(["HealthBoard", "Medicine"], as_index=False)
    .agg(
        TotalQuantity=("Quantity", "sum"),
        TotalCost=("Cost", "sum")
    )
)
# 22.2: keep only top 20 by TotalCost
high_cost_low_top20 = (
    high_cost_low_summary
    .sort_values("TotalCost", ascending=False)
    .head(20)
)

# 22.2: Attach coordinates (reuse your board_coords)
high_cost_low_map = high_cost_low_summary.merge(
    board_coords, on="HealthBoard", how="left"
)
                        # Now high_cost_low_map has:HealthBoard, Medicine, TotalQuantity, TotalCost, HBName, lat, lon

# 22.3: Bubble map – Top 20 High-Cost, Low-Quantity Medicines (Colourful Map)

customdata = high_cost_low_map[["Medicine", "TotalQuantity", "TotalCost"]].to_numpy()

fig_high_cost_low_qty = px.scatter_geo(
    high_cost_low_map,
    lat="lat",
    lon="lon",
    hover_name="HBName",
    size="TotalCost",                # bigger cost → bigger bubble
    color="TotalCost",               # colour by cost (not medicine)
    size_max=60,
    color_continuous_scale="Plasma", # nice purple–yellow scale
    projection="mercator",
)

# Beautiful tooltip
fig_high_cost_low_qty.update_traces(
    customdata=customdata,
    hovertemplate=(
        "<b>%{hovertext}</b><br>"
        "Medicine: %{customdata[0]}<br>"
        "Quantity: %{customdata[1]:,.0f}<br>"
        "Total cost: £%{customdata[2]:,.0f}"
        "<extra></extra>"
    )
)

# Zoom to Scotland + nicer background
fig_high_cost_low_qty.update_geos(
    fitbounds="locations",
    visible=False,
    projection_scale=10,
    center={"lat": 56.8, "lon": -4.0},
    showland=True,
    landcolor="#f5f5f5",
    coastlinecolor="#999999",
)

# Big bold title + good size
fig_high_cost_low_qty.update_layout(
    title={
        "text": "<b>Top 20 High-Cost, Low-Quantity Medicines by NHS Health Board</b>",
        "y": 0.95,
        "x": 0.5,
        "xanchor": "center",
        "yanchor": "top",
        "font": dict(size=22),
    },
    width=1000,
    height=900,
    margin=dict(l=0, r=0, t=90, b=0),
)

fig_high_cost_low_qty.show()
